# **Transformers and XAI: practice**
**Добро пожаловать на практику по трансформерам, друзья!**

Прежде чем приступить, спасибо вам за внимание и упорство в задачах. Ваши отзывы, замечания и идеи делают курс лучше, поэтому делитесь ими в чатах курса.

**Перейдем к уроку.**

Интерпретация моделей является интересной и захватывающей задачей, и трансформеры с их механизмом внимания — не исключение. Работая с ними, важно учитывать **качество предобучения**.

Как правило, если при анализе важности в модели опираться на карты внимания, нужно использовать *хорошо предобученную* модель для вашей задачи. Только у такой модели по картам внимания имеет смысл искать гипотезы о данных, и их потом проверяют другими методами.  

Карты внимания — не единственный инструмент: такие методы, как SHAP, LIME, Integrated Gradients и другие тоже могут быть полезны.

**В этой практике вы:**

- научитесь визуализировать механизм внимания (attention weights);
- примените методы интерпретации (такие как SHAP) для анализа решений, принимаемых моделью;
- познакомитесь с инструментарием, позволяющим визуализировать модели типа transformer в пару строк кода;
- обсудите важность интерпретации моделей в реальных приложениях и разберете практические кейсы.

Приятного кодинга!

Начнем с установки всех нужных библиотек. Не пугайтесь, на это может потребоваться 2-5 минут.

In [ ]:
!pip install transformers==5.18.0 shap==0.52.0 transformers-interpret==0.10.0 ferret-xai==0.4.1 -q # Как всегда подтянем все нужные библиотеки

И импортируем необходимые сущности.

In [ ]:
# MODEL
from transformers import AutoModelForSequenceClassification, AutoTokenizer, pipeline
import torch

# XAI Libs
from transformers_interpret import SequenceClassificationExplainer
from ferret import Benchmark
import shap

# VIZ
import numpy as np
import matplotlib.pyplot as plt


Будем работать над классической и очень красивой задачей — задачей классификации тональности текста. Чтобы не тратить время, возьмем предобученную модель `distilbert`.

**DistilBERT** — это более маленькая и производительная модель, чем базовый BERT (Bidirectional Encoder Representations from Transformers). Она имеет на 40% меньше параметров, чем google-bert/bert-base-uncased и работает на 60% быстрее, сохраняя при этом более 95% производительности оригинальной модели.

In [ ]:
model_name = "distilbert-base-uncased-finetuned-sst-2-english"

model = AutoModelForSequenceClassification.from_pretrained(model_name, attn_implementation="eager")
tokenizer = AutoTokenizer.from_pretrained(model_name)

Определим текст для работы. Вы можете задать также свой, но для выполнения заданий в тренажере используйте оригинальный.


In [ ]:
data = ["You look terrible! Maybe you should rest?"]

In [ ]:
txt_tokens = tokenizer(data[0], return_tensors="pt", add_special_tokens=True) # Токенизируем слова
labels = tokenizer.tokenize(data[0], add_special_tokens=True) # Сохраним labels для построения будущей карты внимания

**Какой токен id соответствует первому слову?**
Ваш ответ здесь

Внимание будем строить на основе взятия медианы по выходам трансформера. По умолчанию количество карт внимания в модели есть **num_layers*num_heads**.

**Проверьте, сколько карт внимания у модели DistilBERT?**

In [ ]:
# Ваш ответ здесь

Поскольку инференс трансформера трудоемкая задача, для получения карт внимания нужно указывать специальный гиперпараметр — `output_attentions=True`. Для визуализации используем медианное значение по всем головам.

**Note:** Ничто не мешает взять среднее, минимум, максимум или другую статистику. Какая будет наиболее информативной, сказать сложно; в этой тетради берем медиану.

In [ ]:
predictions = model(**txt_tokens, output_attentions=True)

In [ ]:
all_attentions = np.array([i.detach().numpy() for i in predictions.attentions[:]]).squeeze(1)

med_layers_att = np.median(all_attentions, axis=0)
med_heads_att = np.median(med_layers_att, axis=0)


In [ ]:
# Визуализируем внимание

fig, ax = plt.subplots(figsize=(6, 12))
ax.imshow(np.median(all_attentions[3], axis=0))


ax.set_xticks(np.arange(len(labels)), labels=labels)
ax.set_yticks(np.arange(len(labels)), labels=labels);


Чтобы сгладить оценку, можно использовать еще одно усреднение, но уже на данном этапе видно, что медианная карта не совсем информативна. Так происходит, потому что каждая голова может брать на себя разные паттерны входной последовательности. Кроме того, модель училась работать с тональностью отзывов, а не с тональностью в целом.

Но минусы не значат, что мы не можем продолжить исследование. К модели типа трансформер могут быть применены также другие методы.


Начнем с shap.

### **shap X transformers**

In [ ]:
classifier = pipeline('text-classification', top_k=None, model=model, tokenizer=tokenizer)
explainer = shap.Explainer(classifier)

print('Predictions')
print(classifier(data))

# Высчитаем значения shap
shap_values = explainer(data)

In [ ]:
shap.plots.text(shap_values)

**Проанализируйте график SHAP (как его читать, разобрано в модуле «Классические методы»). Чему равен прогноз модели?**

**Чему равно base value класса NEGATIVE?**

Рассмотрим еще один тип графика — barplot. Можно визуализировать влияние токенов на «POSITIVE» и «NEGATIVE» классы отдельно.

In [ ]:
# Влияние токенов на Negative класс
shap.plots.bar(shap_values[0, :, "NEGATIVE"], max_display=10)

In [ ]:
# Влияние токенов на Positive класс
shap.plots.bar(shap_values[0, :, "POSITIVE"], max_display=10)

### **transformers-interpret X transformers**

Еще одна механическая реализация способа заглянуть внутрь модели — библиотека `transformers-interpret`. Как вы можете заметить из названия — она разработана специально под модели типа трансформер.

Из плюсов:
- Transformers Interpret позволяет объяснить любую модель transformer всего в две строки
- Доступны объяснения как для текстовых моделей, так и для моделей компьютерного зрения
- Визуализации доступны ​​в виде легко сохраняемых файлов png и html

Из минусов:
- Обертка дает меньше контроля над получением объяснения

В целом, данная библиотека отличный способ получить объяснение быстро. Она позволяет экономить время и, кроме того, облегчает создание отчета по объяснениям за счет удобного сохранения файлов.

Для примеров с MultiLabel, Zero Shot Classification, Question Answering, Token Classification (NER) и Image Classification переходите [сюда](https://github.com/cdpierse/transformers-interpret/tree/master).

In [ ]:
cls_explainer = SequenceClassificationExplainer(model, tokenizer) # согласно философии библиотеки, объявляем классификатор
word_attributions = cls_explainer(data[0]) # получаем атрибуции в пару строк

In [ ]:
cls_explainer.visualize("distilbert_viz.html", true_class='NEGATIVE') # осуществляем визуализацию атрибуций

In [ ]:
fig, ax = plt.subplots()

y_pos = np.arange(len(word_attributions))
word_attributions_numbers = [i[1] for i in word_attributions]
word_attributions_labels = [i[0] for i in word_attributions]

ax.barh(y_pos, word_attributions_numbers)
ax.set_yticks(y_pos, labels=word_attributions_labels)
ax.set_title('Word attributions with transformers-interpret')
plt.show()

### **FERRET x transformers**

Как вы можете заметить, атрибуции, получаемые разными методами в рамках одной модели — различны. Мы видели это на протяжении курса и наблюдаем сейчас. Сказать, что лучше — нельзя. Какой метод выбрать, зависит от вопроса к модели.


Из плюсов фреймворка:
- быстрый сравнительный анализ атрибуций разных методов объяснения
- поддержка аудиоданных и объяснений для них

Из минусов:
- Аналогично transformers-interpret, обертка дает меньше контроля над получением объяснения

In [ ]:
bench = Benchmark(model, tokenizer)

explanations = bench.explain('You look terrible! Maybe you should rest?', target=0)
evaluations = bench.evaluate_explanations(explanations, target=0)

In [ ]:
# bench.show_table(explanations) падает на pandas >= 2: во фразе два токена «you»,
# а ferret 0.4.1 переименовывает повторы через приватный метод pandas, которого больше нет.
# Ту же таблицу соберем из bench.get_dataframe
import pandas as pd
from ferret.benchmark import SCORES_PALETTE

table = bench.get_dataframe(explanations).iloc[:, 1:-1]  # без [CLS] и [SEP], как в show_table
tokens = pd.Series(table.columns)
table.columns = [t if k == 0 else f'{t}.{k}' for t, k in zip(tokens, tokens.groupby(tokens).cumcount())]
table.style.background_gradient(axis=1, cmap=SCORES_PALETTE, vmin=-1, vmax=1).format('{:.2f}')

**Какой токен получил наибольшую оценку у метода Partition SHAP в таблице ferret?** Смотрите строку Partition SHAP: этот метод от запуска к запуску дает одни и те же числа, а оценки LIME меняются, потому что LIME выбирает возмущения входа случайно.

 Еще несколько библиотек для работы с трансформерами — в последнем шаге урока на платформе.

Спасибо за работу, друзья!